In [ ]:
!pip install -q ai-accelerator

In [ ]:
%pip install -q torch pytorch-crf

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

ner_ar = load_dataset("iahlt/arabic_ner_mafat")
ner_ar

In [ ]:
import pandas as pd
df = pd.DataFrame(ner_ar["train"])
df.sample(5)

In [ ]:
import random

cols = df.columns
id = random.choice(df.index)

for col in cols:
  print(col)
  print(df[col][id])
  print("\n")

In [ ]:
entity_types = {
    "O": "Outside of a named entity",
    "PER": "People (real or fictional)",
    "ORG": "Organizations, institutions, companies",
    "GPE": "Geo-political entities (countries, cities, states)",
    "LOC": "Non-GPE locations (mountains, rivers, regions)",
    "FAC": "Facilities (buildings, airports, landmarks)",
    "TIMEX": "Time expressions (dates, periods)",
    "TTL": "Titles or professions (President, Prime Minister)",
    "WOA": "Works of art (books, movies, songs)",
    "EVE": "Events (Olympics, World Cup)",
    "DUC": "Products (Apple, BMW, Coca-Cola)",
    "ANG": "Languages (Arabic, French, Hebrew)",
    "INFORMAL": "Informal/slang expressions",
    "MISC": "Miscellaneous (catch-all category)"
}

In [ ]:
ner = df.drop(columns=["ner_tags","spaces","spans","record","text"])
ner.columns


In [ ]:
ner.sample(5)

In [ ]:
ner.raw_tags.value_counts().to_frame()

In [ ]:
words = []
tags = []
for i in range(len(ner)):
   for word , tag in  zip(ner.tokens[i],ner.raw_tags[i]):
      words.append(word)
      tags.append(tag)
print(f"statistics : ")
print("----------------------------------------------")
print(f"the number of tokens : {len(words)}")
print(f"the number of unique words : {len(set(words))}")
print(f"the number of unique tags : {len(set(tags))}")
print("----------------------------------------------")
print("\n")
df_ner = pd.DataFrame({"words":words,"tags":tags})
df_ner.sample(5)
print("----------------------------------------------")
print(f"the number of each entity tag: \n ")
df_ner.tags.value_counts().to_frame()



In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader , TensorDataset

In [ ]:
from torch.utils.data import Dataset

# Get unique words and tags
unique_words_list = df_ner["words"].unique().tolist()
unique_tags_list = df_ner["tags"].unique().tolist()

# Create mappings from word/tag string to integer index
word2idx = {word: idx for idx, word in enumerate(unique_words_list)}
tag2idx = {tag: idx for idx, tag in enumerate(unique_tags_list)}

# Prepare lists of tensors for DataLoader, one tensor per sentence
all_words_tensors = []
all_tags_one_hot_tensors = []

for i in range(len(ner)):
    # Convert words to numerical indices for the current sample
    current_words_indices = [word2idx[word] for word in ner["tokens"][i]]
    current_words_tensor = torch.tensor(current_words_indices, dtype=torch.long)
    all_words_tensors.append(current_words_tensor)

    # Convert raw_tags to numerical indices and then one-hot for the current sample
    current_tags_indices = [tag2idx[tag] for tag in ner["raw_tags"][i]]
    current_tags_tensor_indexed = torch.tensor(current_tags_indices, dtype=torch.long)
    # Apply F.one_hot to the numerical tag indices
    current_tags_tensor_one_hot = F.one_hot(current_tags_tensor_indexed, num_classes=len(unique_tags_list))
    all_tags_one_hot_tensors.append(current_tags_tensor_one_hot)

# Define a custom Dataset class to handle lists of tensors
class CustomNERDataset(Dataset):
    def __init__(self, words_list, tags_list):
        self.words_list = words_list
        self.tags_list = tags_list

    def __len__(self):
        return len(self.words_list)

    def __getitem__(self, idx):
        return self.words_list[idx], self.tags_list[idx]

# Create an instance of the custom dataset
datasets = CustomNERDataset(all_words_tensors, all_tags_one_hot_tensors)


In [ ]:
import os
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence , pad_sequence



def collate_fn(batch):
    words_tensors_indexed , tags_hot_encoder = zip(*batch)
    words_tensors_padded = pad_sequence(words_tensors_indexed,batch_first=True)
    tags_hot_encoder_padded = pad_sequence(tags_hot_encoder,batch_first=True)
    length_tensor_words = torch.tensor([len(list_words) for list_words in words_tensors_indexed])
    length_tensor_tags = torch.tensor([len(list_tags) for list_tags in tags_hot_encoder])
    words_packed = pack_padded_sequence(words_tensors_padded, length_tensor_words, batch_first=True, enforce_sorted=False)
    tags_packed = pack_padded_sequence(tags_hot_encoder_padded, length_tensor_tags, batch_first=True, enforce_sorted=False)
    return words_packed , tags_packed

dataloader = DataLoader(datasets, batch_size=32,collate_fn= collate_fn , shuffle=True, num_workers=os.cpu_count(), pin_memory=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
  print("GPU is available")
  properties = torch.cuda.get_device_properties(0)
  print(f"The GPU name is : {properties.name}")
  print(f"The GPU memory is : {properties.total_memory / 1024**3} GB")
  print(f"{properties.name} has ~ {properties.total_memory / 1024**3:.2f} GB VRAM. That’s why batch sizes of 32 – 256 should be safe for model")
else:
  print("GPU is not available")

In [ ]:
try:
    import torchcrf
    print("torchcrf successfully imported!")
except ModuleNotFoundError:
    print("Error: torchcrf is still not found. Please ensure the runtime was restarted and all cells, including the pip install, were run.")
    print("installing torchcrf...")
    !pip install  torchcrf

In [ ]:

import torch.nn as nn
from torchcrf import CRF

class NER_Arabic(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, embedding_dim: int, device: torch.device):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=embedding_dim, num_layers=1 , batch_first=True , bidirectional=True)
        self.linear = nn.Linear(in_features= embedding_dim * 2 ,out_features=num_classes)
        self.crf = CRF(num_tags=num_classes, batch_first=True)
        self.device = device


    def forward(self, input_indices, tags=None):
        # Get embeddings from input indices
        embeddings = self.embedding(input_indices)
        # Pass embeddings through the lstm layer
        output_lstm, _ = self.lstm(embeddings)
        emissions = self.linear(output_lstm)

        if tags is not None:
            # If tags are provided, compute the CRF loss (negative log likelihood)
            # CRF.forward expects integer tags, not one-hot
            # We would need `tags` to be integer indices here
            # For now, if we use nn.CrossEntropyLoss externally, we'll just return emissions
            return emissions # Return emissions for external loss calculation
        else:
            # For inference, decode the best path
            # Move emissions to device as CRF might expect it there
            return self.crf.decode(emissions.to(self.device))


In [ ]:
import tqdm.auto  as tqdm
from torch.amp import autocast, GradScaler

EPOCHS = 30
embedding_dim = 300

model = NER_Arabic(vocab_size=len(unique_words_list), num_classes=len(unique_tags_list), embedding_dim=embedding_dim, device="cuda").to(device) # Move model to device
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_classes = len(unique_tags_list)

scaler = GradScaler()

for epoch in tqdm.tqdm(range(EPOCHS)):
    epoch_loss = 0
    for batch_inputs , batch_labels in dataloader:
        # Unpack words_packed
        unpacked_batch_inputs, _ = pad_packed_sequence(batch_inputs, batch_first=True)
        unpacked_batch_inputs = unpacked_batch_inputs.to(device)

        # Unpack tags_packed (batch_labels) and convert from one-hot to class indices
        unpacked_batch_labels, _ = pad_packed_sequence(batch_labels, batch_first=True)
        # Assuming batch_labels were one-hot encoded, convert to class indices for CrossEntropyLoss
        unpacked_batch_labels = torch.argmax(unpacked_batch_labels, dim=-1).to(device)

        optimizer.zero_grad()
        with autocast(device_type='cuda'):
          emissions = model(unpacked_batch_inputs, tags=unpacked_batch_labels) # Pass tags to get emissions

          # compute loss
          # CrossEntropyLoss expects (N, C, d1, d2) and target (N, d1, d2) or (N, C) and (N)
          # Here, emissions will be (batch_size, sequence_length, num_classes)
          # unpacked_batch_labels will be (batch_size, sequence_length)
          loss = loss_fn(emissions.permute(0, 2, 1), unpacked_batch_labels)

        # backprop
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    if epoch % 5 == 0:
        print("\n")

        print("----------------------------------------------")
        print(f"EPOCH {epoch} ------- LOSS: {avg_loss:.4f}")
        print("----------------------------------------------")
        print("\n")

In [ ]:
!pip install -q fasttext

In [ ]:
!pip install -q torchinfo
!pip install -q torchtext


In [ ]:
import torchinfo

summary = torchinfo.summary(model)
summary

In [ ]:
import nltk

# Download necessary NLTK data for Arabic tokenization
nltk.download('punkt')
# Note: 'punkt_tab' is not a standard, directly downloadable package name for nltk.download.
# If this fails, it indicates a deeper issue with NLTK's Arabic punkt tokenizer setup.
print("Attempting to download 'punkt_tab' as explicitly suggested by the error...")
nltk.download('punkt_tab')
print("'punkt_tab' download attempt finished.")

In [ ]:
import nltk

def NERARABIC(sentence):
    # Use nltk.wordpunct_tokenize for Arabic sentences to avoid 'punkt_tab' LookupError
    words = nltk.wordpunct_tokenize(sentence) # This tokenizes based on punctuation and spaces

    # Check if all words are in word2idx. If not, handle OOV words or skip them.

    processed_words = [word for word in words if word in word2idx]

    if not processed_words:
        return "No recognizable words in the vocabulary for this sentence."

    words_indices = [word2idx[word] for word in processed_words]

    # Process the sentence word by word to get NER tags
    results = []
    with torch.inference_mode():
        # Convert list of word indices to a single tensor, add batch dimension
        sentence_tensor = torch.tensor(words_indices, dtype=torch.long).unsqueeze(0).to(device)

        # The model's forward method for inference returns decoded sequence directly
        decoded_tags = model(sentence_tensor)

        # decoded_tags is a list of lists (batch_size, sequence_length)
        # For batch_size=1, we take the first list.
        predicted_tag_indices = decoded_tags[0]

        for word, tag_idx in zip(processed_words, predicted_tag_indices):
            tag = unique_tags_list[tag_idx]
            # Extract the base entity type from the IOB tag (e.g., 'B-PER' -> 'PER')
            if tag == 'O':
                base_entity_type = tag
            else:
                base_entity_type = tag.split('-')[-1]

            # Retrieve the full description from entity_types using the base type
            entity_desc = entity_types.get(base_entity_type, f"Unknown: {base_entity_type}")
            print(f"{word}: {entity_desc}")

        return f"{sentence}"

In [ ]:
new_sentence = "يزور الرئيس الأمريكي جو بايدن المملكة العربية السعودية هذا الأسبوع"
NERARABIC(new_sentence)

*AI Accelerator* is an open-source platform I built to help teams run AI in production with structure and confidence. **It covers the full lifecycle of machine learning operations: deploying models as live services, monitoring health and drift, enforcing governance policies**. The platform is operated through a CLI tool called , which is designed to be run directly in a  **command prompt (CMD) **, giving teams a simple and reliable way to execute commands, track performance, and apply policies without needing complex interfaces.

In [ ]:
!aiac --help

In [ ]:
!aiac server --help

In [ ]:
!aiac deployment --help

In [ ]:
!aiac monitoring --help

In [ ]:
!aiac governance --help